# 01B — Team 16 S3 Structure & Corpus Check

**Joseph (s1602) — Tasks 2 & 3 on my sheet. Run after 01A.**

Small housekeeping notebook. It:
1. Creates our S3 folder structure under the team16 prefix (raw / processed / artifacts / models).
2. Confirms the corpus JSON is in the raw folder (uploads it if I have a local copy).
3. Enables versioning awareness so we can trace which data produced which model.

This gives Ria a clean place to save the labelled table and gives my pipeline predictable paths to read from.

## 0. Config (same values as everywhere)

In [1]:
import boto3
from botocore.exceptions import ClientError

REGION  = "ap-southeast-1"
TEAM_ID = "team16"
BUCKET  = "nyp-26s1-iti113"
PROJECT = "singlish-keyboard"

BASE = f"iti113/{TEAM_ID}/data/{PROJECT}"
RAW_KEY       = f"{BASE}/raw/smsCorpus_en_2015.03.09_all.json"
PROCESSED_KEY = f"{BASE}/processed/sms_labelled.parquet"

s3 = boto3.client("s3", region_name=REGION)
print("Bucket:", BUCKET)
print("Base prefix:", BASE)

Bucket: nyp-26s1-iti113
Base prefix: iti113/team16/data/singlish-keyboard


## 1. Create the folder structure

S3 has no real folders — they're implied by key prefixes. We create tiny placeholder objects so the structure is visible in the console and every stage has a predictable home.

In [2]:
folders = ["raw", "processed", "artifacts", "models", "monitoring"]
for f in folders:
    key = f"{BASE}/{f}/.keep"
    s3.put_object(Bucket=BUCKET, Key=key, Body=b"")
    print("created:", f"s3://{BUCKET}/{BASE}/{f}/")
print("\nStructure ready.")

created: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/raw/
created: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/processed/
created: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/artifacts/
created: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/models/
created: s3://nyp-26s1-iti113/iti113/team16/data/singlish-keyboard/monitoring/

Structure ready.


## 2. Confirm the corpus is in place

Checks whether the corpus JSON is already in the raw folder. If not, and a local copy exists in this Studio space, it uploads it. Adjust `LOCAL_JSON` if your local filename/path differs.

In [3]:
import os

LOCAL_JSON = "smsCorpus_en_2015.03.09_all.json"  # adjust if yours sits elsewhere

def s3_exists(bucket, key):
    try:
        s3.head_object(Bucket=bucket, Key=key); return True
    except ClientError:
        return False

if s3_exists(BUCKET, RAW_KEY):
    size = s3.head_object(Bucket=BUCKET, Key=RAW_KEY)["ContentLength"]
    print(f"Corpus already in S3 ({size/1e6:.1f} MB): {RAW_KEY}")
elif os.path.exists(LOCAL_JSON):
    s3.upload_file(LOCAL_JSON, BUCKET, RAW_KEY)
    print("Uploaded local corpus to:", RAW_KEY)
else:
    print("NOT FOUND locally or in S3.")
    print("Action: upload smsCorpus_en_2015.03.09_all.json into this Studio folder, then re-run this cell.")

Corpus already in S3 (46.0 MB): iti113/team16/data/singlish-keyboard/raw/smsCorpus_en_2015.03.09_all.json


## 3. Verify read access (quick sanity check)

Confirms we can actually read the corpus back — catches permission problems now rather than mid-pipeline.

In [4]:
import json
try:
    obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
    head = obj["Body"].read(2000)  # first 2KB only
    n = head.count(b'"message"')
    print("Read OK. Corpus is reachable from S3.")
    print("Preview (first 200 chars):", head[:200].decode("utf-8", "ignore"))
except ClientError as e:
    print("Read FAILED — permissions or key issue:", e)

Read OK. Corpus is reachable from S3.
Preview (first 200 chars): {"smsCorpus": {"@date": "2015.03.09", "@version": 1.2, "message": [{"@id": 10120, "text": {"$": "Bugis oso near wat..."}, "source": {"srcNumber": {"$": 51}, "phoneModel": {"@manufactuer": "unknown", "


## Done

Structure created, corpus confirmed. Paths for the rest of the pipeline:
- Raw corpus: `RAW_KEY`
- Ria's labelled output (she writes here): `PROCESSED_KEY`
- My model artifacts land under `artifacts/` and `models/`

Next on my sheet: adapt Notebook 03 (the pipeline) against these paths.